In [1]:
# Calculate and save OSDMA8

In [2]:
import os
import xarray as xr

In [3]:
# === Processing function ===
def calculate_maximum_6month_mean(start_year, end_year, monthly_mda8):
    years = list(range(start_year, end_year + 1))

    max_vals = []

    for year in years:
        # Define window: Jan of this year to Mar of next year
        start = f"{year}-01"
        end = f"{year + 1}-03"

        # Subset to this window
        subset = monthly_mda8.sel(time=slice(start, end))

        # Compute 6-month rolling mean along time
        rolling_6m = subset.rolling(time=6, center=False).mean()

        # Find index of maximum
        max_idx = rolling_6m.argmax(dim="time")
        max_val = rolling_6m.isel(time=max_idx)

        # Expand dimensions for consistent output
        max_val = max_val.expand_dims(year=[year])

        max_vals.append(max_val)

    # Combine across years
    annual_max_6m = xr.concat(max_vals, dim="year")

    return annual_max_6m

In [4]:
# === Historical ===

In [5]:
# === Path config ===
BASE_DIR = "/glade/work/awells/air_quality/CESM/ozone/MDA8/"
SAVE_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8/"
SCENARIOS = ["hist"]
ens_num = 1


# === Main loop ===
for scenario in SCENARIOS:
    print(f"Processing {scenario}")
    if scenario == "hist":
        dates = "19900101-20091231"
        # Final year will not be complete due to SH Jan-Mar missing
        new_dates = "1990-2008"
    in_file = f"MDA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
    file_list = [os.path.join(BASE_DIR, in_file)]
    OSDMA8 = []

    for file in file_list:
        if not os.path.exists(file):
            print(f"Missing: {file}")
            continue

        print(f"Reading {os.path.basename(file)}")
        monthly_mda8 = xr.open_dataarray(file)

        # Create list of years to calculate over
        start_year = int(str(monthly_mda8.time.dt.year[0].values))
        # Take second to last year to keep final March
        end_year = int(str(monthly_mda8.time.dt.year[-1].values - 1))

        annual_max_6m = calculate_maximum_6month_mean(start_year, end_year,
                                                      monthly_mda8)

        OSDMA8.append(annual_max_6m)

    if OSDMA8:
        combined = xr.concat(OSDMA8, dim="year")

        out_file = f"OSDMA8_CESM2_{scenario}_{ens_num:02d}_{new_dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving to {out_path}")
        description = ("OSDMA8: Highest seasonal (6-month) average of 8-hour "
                       "daily maximum ozone concentrations across 15 months "
                       "(Jan-Mar) - scripts by A.F. Wells (2025)")
        combined.attrs = monthly_mda8.attrs
        combined.attrs["description"] = description
        combined.attrs["scenario"] = scenario
        combined.to_netcdf(out_path)

print("All processing complete.")

Processing hist
Reading MDA8_CESM2_hist_01_19900101-20091231.nc
Saving to /glade/work/awells/air_quality/CESM/ozone/OSDMA8/OSDMA8_CESM2_hist_01_1990-2008.nc
All processing complete.


In [6]:
# === Future Scenarios ===

In [7]:
# === Path config ===
BASE_DIR = "/glade/work/awells/air_quality/CESM/ozone/MDA8/"
SAVE_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8/"
SCENARIOS = ["ARISE", "SSP245"]

# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "20350101-20691231"
            # Final year will not be complete due to SH Jan-Mar missing
            new_dates = "2035-2068"
        elif scenario == "SSP245":
            dates = "20200101-20691231"
            # Final year will not be complete due to SH Jan-Mar missing
            new_dates = "2020-2068"
        in_file = f"MDA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        file_list = [os.path.join(BASE_DIR, in_file)]
        OSDMA8 = []

        for file in file_list:
            if not os.path.exists(file):
                print(f"Missing: {file}")
                continue

            print(f"Reading {os.path.basename(file)}")
            monthly_mda8 = xr.open_dataarray(file)

            # Create list of years to calculate over
            start_year = int(str(monthly_mda8.time.dt.year[0].values))
            # Take second to last year to keep final March
            end_year = int(str(monthly_mda8.time.dt.year[-1].values - 1))

            annual_max_6m = calculate_maximum_6month_mean(start_year, end_year,
                                                          monthly_mda8)

            OSDMA8.append(annual_max_6m)

        if OSDMA8:
            combined = xr.concat(OSDMA8, dim="year")

            out_file = f"OSDMA8_CESM2_{scenario}_{ens_num:02d}_{new_dates}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving to {out_path}")
            description = ("OSDMA8: Highest seasonal (6-month) average of "
                           "8-hour daily maximum ozone concentrations across "
                           "15 months (Jan-Mar) - scripts by A.F. Wells (2025)")
            combined.attrs = monthly_mda8.attrs
            combined.attrs["description"] = description
            combined.attrs["ensemble_number"] = ens_num
            combined.attrs["scenario"] = scenario
            combined.to_netcdf(out_path)

print("All processing complete.")

Processing ARISE, Ensemble 01
Reading MDA8_CESM2_ARISE_01_20350101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/ozone/OSDMA8/OSDMA8_CESM2_ARISE_01_2035-2068.nc
Processing ARISE, Ensemble 02
Reading MDA8_CESM2_ARISE_02_20350101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/ozone/OSDMA8/OSDMA8_CESM2_ARISE_02_2035-2068.nc
Processing ARISE, Ensemble 03
Reading MDA8_CESM2_ARISE_03_20350101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/ozone/OSDMA8/OSDMA8_CESM2_ARISE_03_2035-2068.nc
Processing ARISE, Ensemble 04
Reading MDA8_CESM2_ARISE_04_20350101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/ozone/OSDMA8/OSDMA8_CESM2_ARISE_04_2035-2068.nc
Processing ARISE, Ensemble 05
Reading MDA8_CESM2_ARISE_05_20350101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/ozone/OSDMA8/OSDMA8_CESM2_ARISE_05_2035-2068.nc
Processing ARISE, Ensemble 06
Reading MDA8_CESM2_ARISE_06_20350101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/ozone/OSDM